# Data Collection Workflow

### Required libraries

In [1]:
!python.exe -m pip install -U pip
!pip install beautifulsoup4 pandas selenium requests

In [2]:
import requests, re, json, time
from bs4 import BeautifulSoup
import pandas as pd
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementNotInteractableException
from bs4 import BeautifulSoup

In [3]:
# Configure session with retry strategy
def create_session():
    session = requests.Session()
    retry_strategy = Retry(
        total=3,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"],  # Updated from method_whitelist
        backoff_factor=1
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

## Load existing data

In [4]:
# Load existing boxers list from txt file
def load_existing_boxers(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            existing_boxers = set(line.strip() for line in file if line.strip())
        return existing_boxers
    except FileNotFoundError:
        return set()
    except UnicodeDecodeError:
        with open(file_path, 'r', encoding='latin1') as file:
            existing_boxers = set(line.strip() for line in file if line.strip())
        return existing_boxers

boxers = load_existing_boxers("../data/active_boxing_fighters.txt")
boxers

{'Agit Kabayel',
 'Alan Picasso Romero',
 'Alberto Puello',
 'Andrew Tabiti',
 'Andy Cruz',
 'Andy Ruiz Jr.',
 'Anthony Cacace',
 'Arnold Barboza Jr.',
 'Artur Beterbiev',
 'Badou Jack',
 'Bakhram Murtazaliev',
 'Brian Norman Jr.',
 'Caleb Plant',
 'Callum Smith',
 'Canelo Álvarez',
 'Carlos Adames',
 'Carlos Cuadras',
 'Chris Billam‑Smith',
 'Chris Eubank Jr.',
 'Christian Mbilli',
 'Daniel Dubois',
 'David Benavidez',
 'David Morrell Jr.',
 'Derek Chisora',
 'Devin Haney',
 'Diego Pacheco',
 'Dillian Whyte',
 'Dmitry Bivol',
 'Edgar Berlanga',
 'Efe Ajagba',
 'Eimantas Stanionis',
 'Elwin Soto',
 'Emanuel Navarrete',
 'Erislandy Lara',
 'Evgeny Romanov',
 'Fabio Wardley',
 'Fernando Martinez',
 'Francisco Rodriguez Jr.',
 'Frank Sanchez',
 'Gary Antuanne Russell',
 'Gervonta Davis',
 'Gilberto Ramirez',
 'Hamzah Sheeraz',
 'Israil Madrimov',
 'Jai Opetaia',
 'Janibek Alimkhanuly',
 'Jared Anderson',
 'Jaron Ennis',
 'Jarrell Miller',
 'Jermall Charlo',
 'Jesse Rodriguez',
 'Joe Joyce

## Retrieval data on Wikipedia

In [5]:
# Function to retrieve Wikipedia page with timeout and retry
def get_page_content(name: str, url: str, session=None, timeout=10):
    if session is None:
        session = create_session()
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        resp = session.get(url, headers=headers, timeout=timeout)
        if resp.status_code != 200:
            print(f"Failed to retrieve page for {name} (status {resp.status_code})")
            return None
        return resp.text
    except requests.exceptions.ConnectTimeout:
        print(f"Connection timeout for {name} - skipping")
        return None
    except requests.exceptions.ReadTimeout:
        print(f"Read timeout for {name} - skipping")
        return None
    except requests.exceptions.ConnectionError:
        print(f"Connection error for {name} - skipping")
        return None
    except Exception as e:
        print(f"Unexpected error for {name}: {str(e)} - skipping")
        return None

# Function to scrape one boxer's data from Wikipedia
def scrape_boxer_info(name: str, session=None):
    if session is None:
        session = create_session()
    
    # Construct Wikipedia URL (replace spaces with underscores)
    url = "https://en.wikipedia.org/wiki/" + name.replace(' ', '_')
    page_content = get_page_content(name, url, session)
    if not page_content:
        return None
    
    soup = BeautifulSoup(page_content, 'html.parser')
    
    # Find the infobox table in the page
    infobox = soup.find("table", {"class": "infobox"})
    if infobox is None:
        print(f"No infobox found for {name} – skipping.")
        return None
    
    data = {"Name": name}
    # Go through each table row in the infobox
    for row in infobox.find_all("tr"):
        header = row.find("th")
        value = row.find("td")
        if not header or not value:
            continue  # skip rows that are not "header: value" pairs (e.g. section headers)
        field = header.get_text(strip=True)
        val_text = value.get_text(" ", strip=True)  # get text inside td
        
        # Extract relevant fields
        if field == "Weight":
            weight_match = re.search(r'(\d+(\.\d+)?)\s*(kg|lbs)', val_text)
            if weight_match:
                weight_value = weight_match.group(1)
                weight_unit = weight_match.group(3).lower()
                data["Weight"] = f"{weight_value} {weight_unit}"
        elif field == "Born" or field == "Date of birth":
            born_match = re.search(r'(\d{1,2}\s\w+\s\d{4})', val_text)
            # born_match examples: Teófimo Andrés López Rivera July 30, 1997 (age 27) New York City, U.S.
            # obtain the date of birth only, ignoring age and location
            date_birth_match = re.search(r'(\d{1,2}\s\w+\s\d{4})', val_text)            
            # if born_match:
            #     data["Born"] = born_match.group(1)
            if date_birth_match:
                data["Date_Birth"] = date_birth_match.group(1) if date_birth_match else born_match.group(1)
        elif field == "Height":
            # Example: "5 ft 8 in (173 cm)"
            height_ft_in_match = re.search(r'(\d+)\s*ft\s*(\d+)\s*in', val_text)
            height_cm_match = re.search(r'(\d+)\s*cm', val_text)
            if height_ft_in_match:
                data["Height (ft)"] = f"{height_ft_in_match.group(1)} ft {height_ft_in_match.group(2)} in"
            elif height_cm_match:
                data["Height (cm)"] = f"{height_cm_match.group(1)}"
            else:
                # fallback: extract any number (sometimes only feet or cm is present)
                height_match = re.search(r'(\d+(\.\d+)?)', val_text)
                if height_match:
                    data["Height"] = height_match.group(0)  # Corrigé: utiliser group(0) directement ici
        elif field == "Reach":
            reach_match = re.search(r'(\d+\'\d+\"|\d+\.\d+|\d+)', val_text)
            
            if reach_match:
                # Extract reach in cm and inches
                reach_cm_match = re.search(r'(\d+)\s*cm', val_text)
                reach_inches_match = re.search(r'(\d+)\s*in', val_text)
                
                if reach_cm_match:
                    data["Reach (cm)"] = reach_cm_match.group(1)
                else:
                    data["Reach (cm)"] = None
                
                if reach_inches_match:
                    data["Reach (inches)"] = reach_inches_match.group(1)
                else:
                    data["Reach (inches)"] = None

        elif field == "Stance":
            stance_match = re.search(r'(\w+)', val_text)
            if stance_match:
                data["Stance"] = stance_match.group(1)
        elif field == "Total fights":
            fights_match = re.search(r'(\d+)', val_text)
            if fights_match:
                data["Total fights"] = fights_match.group(1)
        elif field == "Wins":
            wins_match = re.search(r'(\d+)', val_text)
            if wins_match:
                data["Wins"] = wins_match.group(1)
        elif field == "Losses":
            losses_match = re.search(r'(\d+)', val_text)
            if losses_match:
                data["Losses"] = losses_match.group(1)
        elif field == "Draws":
            draws_match = re.search(r'(\d+)', val_text)
            if draws_match:
                data["Draws"] = draws_match.group(1)
        elif field == "No contests":
            no_contests_match = re.search(r'(\d+)', val_text)
            if no_contests_match:
                data["No contests"] = no_contests_match.group(1)
    
        # data[field] = val_text  # Default assignment for other fields
            
    # If age wasn't in Born (e.g., deceased boxers won't have an age there), check for Died field
    if "Age" not in data:
        died_field = infobox.find("th", string="Died")
        if died_field:
            died_text = died_field.find_next("td").get_text(" ", strip=True)
            age_match = re.search(r'\(aged\s+(\d+)\)', died_text)
            if age_match:
                data["Age"] = age_match.group(1)  # age at death
    print(f"Scraped data for {name}: {data}")
    return data

# Function to scrape data for all boxers in the list with progress tracking
def dataset_by_scraping(boxers: set[str]):
    dataset = []
    session = create_session()  # Reuse session for better performance
    total_boxers = len(boxers)
    successful = 0
    failed = 0
    
    print(f"Starting to scrape {total_boxers} boxers from Wikipedia...")
    
    for i, name in enumerate(boxers, 1):
        
        print(f"\n[{i}/{total_boxers}] Processing: {name}")
        
        info = scrape_boxer_info(
                name,
                session,
               )
        
        if info:
            dataset.append(info)
            successful += 1
        else:
            failed += 1
        
        # Add small delay to be respectful to Wikipedia
        time.sleep(1)
        
        # Progress report every 10 boxers
        if i % 10 == 0:
            print(f"\nProgress: {i}/{total_boxers} processed ({successful} successful, {failed} failed)")
    
    print(f"\nScraping completed! Total: {successful} successful, {failed} failed")
    return dataset

In [6]:
# Test avec un échantillon réduit pour vérifier que l'erreur est résolue
print("🧪 Scraping Wikipedia avec échantillon réduit...")

test_dataset = dataset_by_scraping(boxers)

print(f"\n✅ Terminé! {len(test_dataset)} boxeurs scrapés avec succès.")
if test_dataset:
    print("📊 Exemple de données extraites:")
    for key, value in test_dataset[0].items():
        print(f"   {key}: {value}")
        
# Si le test réussit, on peut lancer le scraping complet
if len(test_dataset) > 0:
    print("\n🎉 Test réussi! Prêt pour le scraping complet.")
    print("Pour lancer le scraping complet, décommentez la ligne suivante:")
    print("# dataset = dataset_by_scraping(boxers)")
else:
    print("❌ Test échoué. Vérifiez les erreurs ci-dessus.")

🧪 Scraping Wikipedia avec échantillon réduit...
Starting to scrape 100 boxers from Wikipedia...

[1/100] Processing: Andy Ruiz Jr.
Scraped data for Andy Ruiz Jr.: {'Name': 'Andy Ruiz Jr.', 'Height (ft)': '6 ft 2 in', 'Reach (cm)': '188', 'Reach (inches)': '74', 'Stance': 'Orthodox', 'Total fights': '38', 'Wins': '35', 'Losses': '2', 'Draws': '1'}

[2/100] Processing: Frank Sanchez
No infobox found for Frank Sanchez – skipping.

[3/100] Processing: Dillian Whyte
Scraped data for Dillian Whyte: {'Name': 'Dillian Whyte', 'Date_Birth': '11 April 1988', 'Height (ft)': '6 ft 4 in', 'Reach (cm)': '198', 'Reach (inches)': '78', 'Stance': 'Orthodox', 'Wins': '1', 'Losses': '0'}

[4/100] Processing: Alan Picasso Romero
Scraped data for Alan Picasso Romero: {'Name': 'Alan Picasso Romero', 'Height (ft)': '5 ft 8 in', 'Reach (cm)': '178', 'Reach (inches)': '70', 'Stance': 'Orthodox', 'Total fights': '32', 'Wins': '31', 'Losses': '0', 'Draws': '1'}

[5/100] Processing: Gary Antuanne Russell
Scraped 

In [7]:
dataset_dataframe = pd.DataFrame(test_dataset)
dataset_dataframe

,Name,Height (ft),Reach (cm),Reach (inches),Stance,Total fights,Wins,Losses,Draws,Date_Birth,Height (cm),No contests,Age,Height
0,Andy Ruiz Jr.,6 ft 2 in,188,74,Orthodox,38,35,2,1,NaN,NaN,NaN,NaN,NaN
1,Dillian Whyte,6 ft 4 in,198,78,Orthodox,NaN,1,0,NaN,11 April 1988,NaN,NaN,NaN,NaN
2,Alan Picasso Romero,5 ft 8 in,178,70,Orthodox,32,31,0,1,NaN,NaN,NaN,NaN,NaN
3,Gary Antuanne Russell,5 ft 10 in,175,69,Southpaw,19,18,1,NaN,NaN,NaN,NaN,NaN,NaN
4,Gilberto Ramirez,NaN,191,75,Southpaw,49,48,1,NaN,19 June 1991,189,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,Christian Mbilli,NaN,183,72,Orthodox,29,29,NaN,NaN,26 April 1995,174,NaN,NaN,NaN
77,Arnold Barboza Jr.,5 ft 9 in,183,72,Orthodox,33,32,1,NaN,NaN,NaN,NaN,NaN,NaN
78,Elwin Soto,5 ft 3 in,NaN,NaN,Orthodox,24,21,3,NaN,NaN,NaN,NaN,NaN,NaN
79,Sandor Martín,5 ft 7 in,175,69,Southpaw,46,42,4,NaN,22 August 1993,NaN,NaN,NaN,NaN


#### Save data

In [8]:
dataset_dataframe.to_csv("../data/1_boxers_dataset.csv", index=False)

## Retrieval data on [BoxRec](/data/active_boxing_fighters.txt)